# Distance-Guided Meta-Learning for Molecular Activity Prediction

This notebook reproduces THEMAP's meta-learning workflow:

1. **Distances** between chemical datasets are pre-computed and saved (here, `output/molecule_distances.csv`).
2. For a chosen **target** dataset, select its **k-nearest source** datasets from that distance file.
3. **Meta-train** a Prototypical Network and a from-scratch MAML learner on those source datasets.
4. Evaluate the **low-data gain**: adapt each meta-learned model to *N* ∈ {16, 32, 64, 128} target
   samples and measure **AUROC** on the held-out remainder, versus an MLP trained from scratch on the
   same *N* samples.

The hypothesis: when sources are *close* to the target, meta-learning should beat a from-scratch
baseline most in the lowest-data regime.

### Making support set size a meaningful axis

Two methodology choices keep the *N* sweep honest (set via `TRAIN_SHOT_MODE` and `QUERY_FRACTION`):

- **Matched training shot** (`TRAIN_SHOT_MODE="match"`): a *separate* model is meta-trained for each
  support size, with a training shot that tracks *N* (capped to what the sources can supply). A model
  evaluated at 128-shot was therefore actually trained near the 128-shot regime — not at a single
  fixed 8-shot setting reused for every *N* (the legacy `"fixed"` behaviour, which makes the curves
  look flat because the model was never optimised for larger support).
- **Fixed, comparable query set** (`QUERY_FRACTION=0.5`): a fixed half of the target is held out as
  the query set, shared across *all* support sizes (and support sets are *nested*). AUROC is thus
  measured on the same molecules at every *N*, so the from-scratch baseline's improvement with more
  data is visible instead of being masked by a shrinking, overlapping query set. Support sizes the
  target cannot supply (after reserving the query set) are skipped with a warning.

A caveat to read the curves correctly: **ProtoNet adapts with a frozen encoder** — its "adaptation"
is just the per-class mean embedding — so it *saturates* after a handful of support points and stays
relatively flat by construction. MAML and the from-scratch baseline are the curves expected to move
most with *N*.

In [ ]:
import os
import sys

# Run from the repository root (mirrors the other notebooks in this folder).
repo_path = os.path.dirname(os.path.abspath(""))
os.chdir(repo_path)
sys.path.insert(0, repo_path)

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from themap.metalearning import select_k_nearest_sources  # noqa: E402
from themap.metalearning.config import ExperimentConfig, MAMLConfig, TrainConfig  # noqa: E402
from themap.metalearning.evaluation import LowDataEvaluator  # noqa: E402
from themap.metalearning.runner import MetaLearnExperiment  # noqa: E402
from themap.utils import set_plot_style  # noqa: E402

set_plot_style()
plt.rcParams["figure.figsize"] = (10, 5)

## Configuration

In [ ]:
DATA_DIR = "datasets/"
DISTANCE_FILE = "output/molecule_distances.csv"
TARGET_ID = "CHEMBL1963831"  # a target present in the distance file (test fold)
K = 3  # number of nearest source datasets to meta-train on
FEATURIZER = "ecfp"
SUPPORT_SIZES = [16, 32, 64, 128]
SEEDS = 3  # repeated stratified draws per support size

# How the meta-training shot relates to the evaluation support size N:
#   "match" — train a *fresh* model per support size whose training shot tracks N
#             (capped to what the sources can supply). This is what makes support set
#             size a meaningful axis: a model evaluated at 128-shot was actually trained
#             near the 128-shot regime, not at a single fixed setting reused for every N.
#   "fixed" — FS-Mol single-model protocol: train ONE model once at TRAIN["n_support"]
#             and evaluate it across all support sizes. Set TRAIN["n_support"]=64 to
#             reproduce FS-Mol's ProtoNet/MAML setup (fast: one training run total).
TRAIN_SHOT_MODE = "match"

# Fraction of the target held out as a *fixed* query set, shared across all support
# sizes so AUROC is measured on the same molecules at every N (and support sets are
# nested). With this fixed, the from-scratch baseline rises with N as expected,
# instead of being masked by a shrinking/overlapping query set.
QUERY_FRACTION = 0.5

# Modest meta-training budget so the notebook runs in a few minutes on CPU.
# Note: "match" mode trains one model per support size, so wall-clock scales with
# len(SUPPORT_SIZES). Scale NUM_EPOCHS / EPISODES_PER_EPOCH up for stronger learners.
TRAIN = dict(
    num_epochs=20,
    episodes_per_epoch=40,
    meta_batch_size=8,
    n_support=8,
    n_query=10,
    val_episodes=30,
    patience=8,
    device="auto",
)

## Step 1 — Select the k-nearest source datasets

We read the saved distance file and pick the `K` sources with the smallest distance to the target.

In [ ]:
selected = select_k_nearest_sources(DISTANCE_FILE, TARGET_ID, k=K)
selected_df = pd.DataFrame(selected, columns=["source_id", "distance"])
print(f"Nearest {K} sources to target {TARGET_ID}:")
display(selected_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(selected_df["source_id"], selected_df["distance"], color="#66c2a5")
ax.invert_yaxis()
ax.set_xlabel("Distance to target")
ax.set_title(f"k-nearest source datasets for {TARGET_ID}")
plt.tight_layout()
plt.show()

## Step 2 & 3 — Meta-train and evaluate ProtoNet and MAML

`MetaLearnExperiment.run()` performs the full pipeline for one algorithm: select sources, featurize
once, meta-train, then sweep the low-data support sizes on the target (meta-adapted vs from-scratch
baseline). We run it for both algorithms.

In [ ]:
def run_experiment(algorithm):
    config = ExperimentConfig(
        data_dir=DATA_DIR,
        distance_file=DISTANCE_FILE,
        target_id=TARGET_ID,
        k=K,
        algorithm=algorithm,
        featurizer=FEATURIZER,
        support_sizes=SUPPORT_SIZES,
        train_shot_mode=TRAIN_SHOT_MODE,
        query_fraction=QUERY_FRACTION,
        seeds=SEEDS,
        output_dir=f"metalearn_out/{algorithm}",
        maml=MAMLConfig(inner_lr=0.01, inner_steps=5, eval_inner_steps=16),
        train=TrainConfig(**TRAIN),
    )
    return MetaLearnExperiment(config).run()


proto_results = run_experiment("proto")
maml_results = run_experiment("maml")
proto_results.head()

## Step 4 — Compare meta-learning against the from-scratch baseline

The evaluator reports **AUROC**, **AUPRC** (average precision), and **ΔAUPRC**
(`average_precision − fraction_positive`) per support size. ΔAUPRC is FS-Mol's headline metric — it
corrects for class imbalance, so it is the column to compare against the downloaded
`datasets/fsmol_hardness/FSMol_Eval_ProtoNet/` tables. The plots below use AUROC; the summary table
also carries `delta_auprc_mean`.

In [ ]:
def agg(df, method):
    sub = df[df["method"] == method]
    g = sub.groupby("support_size")["auroc"]
    mean = g.mean()
    # 95% CI half-width (t-interval).
    from scipy import stats

    ci = g.apply(lambda x: stats.sem(x) * stats.t.ppf(0.975, len(x) - 1) if len(x) > 1 else np.nan)
    return mean, ci


series = {
    "ProtoNet (meta)": agg(proto_results, "meta"),
    "MAML (meta)": agg(maml_results, "meta"),
    "Baseline (from scratch)": agg(proto_results, "baseline"),
}

fig, ax = plt.subplots(figsize=(9, 5))
for label, (mean, ci) in series.items():
    ax.plot(mean.index, mean.values, marker="o", label=label)
    ax.fill_between(mean.index, mean.values - ci.values, mean.values + ci.values, alpha=0.15)
ax.set_xscale("log", base=2)
ax.set_xticks(SUPPORT_SIZES)
ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.set_xlabel("Target support set size (N)")
ax.set_ylabel("AUROC on held-out target")
ax.set_title(f"Low-data meta-learning on {TARGET_ID} (k={K} nearest sources)")
ax.legend()
plt.tight_layout()
plt.show()

### Meta-learning gain over the baseline (positive = meta-learning helps)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
base_mean = series["Baseline (from scratch)"][0]
for label in ["ProtoNet (meta)", "MAML (meta)"]:
    mean = series[label][0]
    ax.plot(mean.index, (mean - base_mean).values, marker="s", label=label)
ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
ax.set_xscale("log", base=2)
ax.set_xticks(SUPPORT_SIZES)
ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.set_xlabel("Target support set size (N)")
ax.set_ylabel("AUROC gain vs baseline")
ax.set_title("Does meta-learning help, and where most?")
ax.legend()
plt.tight_layout()
plt.show()

# Tabular summary.
summary = pd.concat(
    [
        LowDataEvaluator.summarize(proto_results),
        LowDataEvaluator.summarize(maml_results),
    ],
    ignore_index=True,
)
summary

## Step 5 — Does distance predict transfer? A close-vs-far contrast

The target above (`CHEMBL1963831`) has **far** nearest sources (distance ≈ 2+), and meta-learning
there tends to *underperform* the from-scratch baseline — a case of **negative transfer**.

THEMAP's premise is that *distance predicts transferability*. To test that directly we build a
**controlled** comparison: take a single assay, hold out part of it as a target, and split the rest
into source shards. By construction those shards are drawn from the **same distribution** as the
target, so their distance is small. We mix in the other (unrelated) datasets as **distant decoys**,
compute a real distance file, and let the k-NN selector choose.

In [ ]:
import gzip
import json
import shutil
from pathlib import Path

from sklearn.model_selection import StratifiedKFold

from themap.data.loader import DatasetLoader
from themap.pipeline import quick_distance

# Controlled demo lives under the gitignored metalearn_out/ directory.
DEMO = Path("metalearn_out/contrast_demo")
DEMO_DATA = DEMO / "datasets"
for sub in ("train", "test"):
    (DEMO_DATA / sub).mkdir(parents=True, exist_ok=True)


def _write_jsonl(path, smiles, labels, assay):
    with gzip.open(path, "wt") as fh:
        for s, y in zip(smiles, labels):
            fh.write(json.dumps({"SMILES": s, "Property": f"{float(y)}", "Assay_ID": assay}) + "\n")


# Shard one balanced assay into a target + 3 same-distribution ("close") source shards.
base = DatasetLoader(DATA_DIR).load_dataset("train", "CHEMBL1613776")
sm = np.array(base.smiles_list)
yy = np.array(base.labels)
folds = [te for _, te in StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(sm, yy)]
_write_jsonl(DEMO_DATA / "test" / "CLOSE_TARGET.jsonl.gz", sm[folds[0]], yy[folds[0]], "CLOSE_TARGET")
for i in range(1, 4):
    _write_jsonl(DEMO_DATA / "train" / f"CLOSE_S{i}.jsonl.gz", sm[folds[i]], yy[folds[i]], f"CLOSE_S{i}")

# Mix in unrelated real datasets as distant decoys.
for tid in ["CHEMBL1023359", "CHEMBL2218944", "CHEMBL2219012", "CHEMBL1614274", "CHEMBL3705844"]:
    shutil.copy(f"{DATA_DIR}/train/{tid}.jsonl.gz", DEMO_DATA / "train" / f"{tid}.jsonl.gz")

# Compute a real distance file over the demo (sources = train fold, target = test fold).
quick_distance(
    str(DEMO_DATA),
    output_dir=str(DEMO / "output"),
    molecule_featurizer=FEATURIZER,
    molecule_method="euclidean",
)
DEMO_DIST = str(DEMO / "output" / "molecule_distances.csv")

ranked = select_k_nearest_sources(DEMO_DIST, "CLOSE_TARGET", k=8)
ranked_df = pd.DataFrame(ranked, columns=["source_id", "distance"])
ranked_df["kind"] = np.where(ranked_df["source_id"].str.startswith("CLOSE"), "close shard", "distant decoy")
display(ranked_df)

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ["#66c2a5" if k == "close shard" else "#fc8d62" for k in ranked_df["kind"]]
ax.barh(ranked_df["source_id"], ranked_df["distance"], color=bar_colors)
ax.invert_yaxis()
ax.set_xlabel("Distance to CLOSE_TARGET")
ax.set_title("Distance-guided selection: close shards vs distant decoys")
plt.tight_layout()
plt.show()

In [ ]:
def run_on(data_dir, distance_file, target_id, algorithm):
    config = ExperimentConfig(
        data_dir=data_dir,
        distance_file=distance_file,
        target_id=target_id,
        k=K,
        algorithm=algorithm,
        featurizer=FEATURIZER,
        support_sizes=SUPPORT_SIZES,
        train_shot_mode=TRAIN_SHOT_MODE,
        query_fraction=QUERY_FRACTION,
        seeds=SEEDS,
        output_dir=None,
        maml=MAMLConfig(inner_lr=0.01, inner_steps=5, eval_inner_steps=16),
        train=TrainConfig(**TRAIN),
    )
    return MetaLearnExperiment(config).run()


# k=3 selects exactly the 3 close shards. (The far-source results were computed
# earlier as proto_results / maml_results on the same support sizes.)
close_proto = run_on(str(DEMO_DATA), DEMO_DIST, "CLOSE_TARGET", "proto")
close_maml = run_on(str(DEMO_DATA), DEMO_DIST, "CLOSE_TARGET", "maml")

In [ ]:
def gain(df):
    m, _ = agg(df, "meta")
    b, _ = agg(df, "baseline")
    return m - b


contrast = {
    "ProtoNet": {
        "far sources (CHEMBL1963831)": gain(proto_results),
        "close sources (shards)": gain(close_proto),
    },
    "MAML": {"far sources (CHEMBL1963831)": gain(maml_results), "close sources (shards)": gain(close_maml)},
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, (algo, lines) in zip(axes, contrast.items()):
    for label, g in lines.items():
        ax.plot(g.index, g.values, marker="o", label=label)
    ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
    ax.set_xscale("log", base=2)
    ax.set_xticks(SUPPORT_SIZES)
    ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel("Target support set size (N)")
    ax.set_title(algo)
    ax.legend()
axes[0].set_ylabel("AUROC gain vs baseline (meta − baseline)")
fig.suptitle("Distance predicts transfer: close sources help, far sources hurt")
plt.tight_layout()
plt.show()

**Same code, same hyperparameters — only the source–target distance differs.** With *far* sources
the gain sits at or below zero (negative transfer); with *close* sources it turns positive, strongest
in the lowest-data regime. This is exactly THEMAP's hypothesis: the distance to candidate source
datasets predicts how much meta-learning will help — which is what makes the distance-guided source
*selection* worthwhile.

## Reproduce from the command line

The same experiment is available as a CLI command:

```bash
themap metalearn datasets/ \
    --distance-file output/molecule_distances.csv \
    --target-id CHEMBL1963831 --k 3 \
    --algorithm proto \
    --support-sizes 16,32,64,128 --seeds 5 \
    --num-epochs 50 -o metalearn_out/proto
```

Swap `--algorithm maml` for the MAML variant. Results (long-form `results.csv`, aggregated
`summary.csv`, selected sources, and training history) are written to the output directory.

**Takeaways to look for:** with sources genuinely close to the target, the meta-learned models
should sit above the from-scratch baseline, with the largest gain at the smallest support sizes —
exactly the low-data regime where transferring knowledge from related datasets matters most.